# FinFact-BD mDeBERTa Quality Filter

**Single-model NLI filtering for Bengali financial misinformation detection.**

## Pipeline
1. Load rule-filtered perturbations from zstd CSV
2. Score each perturbation using mDeBERTa-v3 multilingual NLI
3. Keep samples with contradiction probability ≥ 0.4

## Setup
1. Upload `finfact_bd_perturbed_rule_filtered.csv.zst` as a Kaggle Dataset
2. Add that dataset to this notebook's input
3. Run all cells
4. Download outputs from `/kaggle/working/`

## Outputs
- `finfact_bd_perturbed_filtered.csv` — quality-filtered perturbations
- `mdeberta_scores.json` — per-sample NLI scores
- `mdeberta_report.json` — summary statistics

In [ ]:
# Cell 1: Install dependencies
!pip install zstandard datasets transformers[torch] scikit-learn accelerate -q
!nvidia-smi

In [ ]:
# Cell 2: Imports and paths
import csv, io, json, time, os
from pathlib import Path
from collections import Counter

import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Kaggle paths
INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Cell 3: Load rule-filtered perturbations
import zstandard as zstd

def find_input_file():
    """Find the uploaded rule-filtered CSV (plain or zstd)."""
    for d in INPUT_DIR.iterdir():
        if not d.is_dir():
            continue
        for f in d.iterdir():
            if "rule_filtered" in f.name and f.suffix == ".csv":
                return f
            if "rule_filtered" in f.name and f.suffix == ".zst":
                return f
    raise FileNotFoundError(
        "Upload finfact_bd_perturbed_rule_filtered.csv.zst as a Kaggle Dataset."
    )

def load_csv(path):
    samples = []
    if path.suffix == ".zst":
        with open(path, "rb") as f:
            dctx = zstd.ZstdDecompressor()
            with dctx.stream_reader(f) as reader:
                text_stream = io.TextIOWrapper(reader, encoding="utf-8")
                for row in csv.DictReader(text_stream):
                    samples.append(row)
    else:
        with open(path, "r", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                samples.append(row)
    return samples

INPUT_FILE = find_input_file()
perturbed = load_csv(INPUT_FILE)
print(f"Loaded {len(perturbed)} rule-filtered perturbations from {INPUT_FILE.name}")
print(f"Columns: {list(perturbed[0].keys())}")

# Show distribution
type_counts = Counter(row["perturbation_type"] for row in perturbed)
for ptype, count in type_counts.most_common():
    print(f"  {ptype}: {count}")

In [ ]:
# Cell 4: Load mDeBERTa model
MODEL_ID = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
MAX_LEN = 512
BATCH_SIZE = 32
CONTRADICTION_THRESHOLD = 0.4

# mDeBERTa label mapping: 0=contradiction, 1=entailment, 2=neutral
LABEL_MAP = {0: "contradiction", 1: "entailment", 2: "neutral"}

print(f"Loading mDeBERTa ({MODEL_ID})...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
model = model.to(DEVICE)
model.eval()
print(f"  GPU mem: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"Model loaded on {DEVICE}")

In [ ]:
# Cell 5: Score pairs and filter
def score_batch(pairs):
    """Score a batch using mDeBERTa NLI model."""
    inputs = tokenizer(
        [p[0] for p in pairs],
        [p[1] for p in pairs],
        padding=True, truncation=True,
        max_length=MAX_LEN, return_tensors="pt",
    ).to(DEVICE)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)
    return probs.cpu().numpy()

# Build premise-hypothesis pairs
pairs = [(row["original_text"], row["text"]) for row in perturbed]
print(f"Scoring {len(pairs)} pairs with mDeBERTa ({BATCH_SIZE}/batch)...")

all_probs = []
t0 = time.time()
for i in range(0, len(pairs), BATCH_SIZE):
    batch = pairs[i : i + BATCH_SIZE]
    probs = score_batch(batch)
    all_probs.append(probs)
    if (i // BATCH_SIZE) % 10 == 0:
        elapsed = time.time() - t0
        pct = min(100, 100 * (i + len(batch)) / len(pairs))
        rate = (i + len(batch)) / elapsed if elapsed > 0 else 0
        eta = (len(pairs) - i - len(batch)) / rate if rate > 0 else 0
        print(f"  [{pct:.0f}%] {i+len(batch)}/{len(pairs)} — {elapsed:.0f}s elapsed, ~{eta:.0f}s remaining")

all_probs = np.concatenate(all_probs, axis=0)
elapsed = time.time() - t0
print(f"\nScoring complete in {elapsed:.0f}s ({elapsed/len(pairs)*1000:.1f}ms/sample)")

# Build results
all_results = []
for i in range(len(pairs)):
    pred_idx = int(all_probs[i].argmax())
    all_results.append({
        "label": LABEL_MAP[pred_idx],
        "probs": {
            "contradiction": float(all_probs[i][0]),
            "entailment": float(all_probs[i][1]),
            "neutral": float(all_probs[i][2]),
        },
    })

# Filter
kept = []
discarded = 0
for row, result in zip(perturbed, all_results):
    contra_prob = result["probs"]["contradiction"]
    if contra_prob >= CONTRADICTION_THRESHOLD:
        row["contradiction_score"] = f"{contra_prob:.4f}"
        row["mdeberta_label"] = result["label"]
        kept.append(row)
    else:
        discarded += 1

print(f"\nKept: {len(kept)}, Discarded: {discarded} ({100*discarded/len(perturbed):.1f}%)")

kept_types = Counter(row["perturbation_type"] for row in kept)
print("\nKept per type:")
for ptype, count in kept_types.most_common():
    print(f"  {ptype}: {count}")

# Save filtered CSV
csv_fields = [k for k in kept[0].keys()]
out_path = OUTPUT_DIR / "finfact_bd_perturbed_filtered.csv"
with open(out_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=csv_fields)
    writer.writeheader()
    writer.writerows(kept)
print(f"\nSaved filtered CSV: {out_path}")

# Save scores
scores = []
for row, result in zip(perturbed, all_results):
    scores.append({
        "article_id": row["article_id"],
        "perturbation_type": row["perturbation_type"],
        "label": result["label"],
        "contradiction_prob": result["probs"]["contradiction"],
        "entailment_prob": result["probs"]["entailment"],
        "neutral_prob": result["probs"]["neutral"],
    })
scores_path = OUTPUT_DIR / "mdeberta_scores.json"
with open(scores_path, "w", encoding="utf-8") as f:
    json.dump(scores, f, indent=2, ensure_ascii=False)
print(f"Saved scores: {scores_path}")

# Save report
contra_probs = [r["probs"]["contradiction"] for r in all_results]
report = {
    "model": MODEL_ID,
    "total_samples": len(perturbed),
    "kept": len(kept),
    "discarded": discarded,
    "keep_rate": len(kept) / len(perturbed),
    "threshold": CONTRADICTION_THRESHOLD,
    "contradiction_stats": {
        "mean": float(np.mean(contra_probs)),
        "median": float(np.median(contra_probs)),
        "min": float(np.min(contra_probs)),
        "max": float(np.max(contra_probs)),
        "std": float(np.std(contra_probs)),
    },
    "per_type_keep_rate": {},
}

# Per-type analysis
type_results = {}
for row, result in zip(perturbed, all_results):
    ptype = row["perturbation_type"]
    type_results.setdefault(ptype, Counter())[result["label"]] += 1

print("\nPer-type distribution:")
for ptype, counts in sorted(type_results.items()):
    total = sum(counts.values())
    contra = 100 * counts.get("contradiction", 0) / total
    entail = 100 * counts.get("entailment", 0) / total
    report["per_type_keep_rate"][ptype] = counts.get("contradiction", 0) / total
    print(f"  {ptype}: contradiction={contra:.0f}% entailment={entail:.0f}% (n={total})")

report_path = OUTPUT_DIR / "mdeberta_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(f"Saved report: {report_path}")

In [ ]:
# Cell 6: Summary statistics and output listing
print("=" * 60)
print("mDEBERTA QUALITY FILTER — COMPLETE")
print("=" * 60)
print(f"Input:        {len(perturbed)} rule-filtered perturbations")
print(f"Output:       {len(kept)} quality-filtered perturbations")
print(f"Discarded:    {discarded} ({100*discarded/len(perturbed):.1f}%)")
print(f"Threshold:    {CONTRADICTION_THRESHOLD}")
print(f"Model:        mDeBERTa-v3-base-xnli-multilingual-nli-2mil7")
print(f"Total time:   {elapsed:.0f}s")
print()
print("Files to download from /kaggle/working/:")
print(f"  1. finfact_bd_perturbed_filtered.csv")
print(f"  2. mdeberta_scores.json")
print(f"  3. mdeberta_report.json")
print("=" * 60)